# PhysDrive CPU pilot — Kaggle özel defteri

Bu defter **yalnızca CPU** kullanır. Kaggle'da **Private Dataset** olarak eklenmiş PhysDrive ZIP'ini veya Kaggle'ın ZIP'i açtıktan sonra bağladığı klasörü bulur. HF token gerekmez; ZIP'i Hugging Face'e ayrıca yüklemiyoruz.

**Kaggle'a veri ekleme:** Datasets → New Dataset → bilgisayardan `PhysDrive(Pub) (1).zip` dosyasını seç → görünürlüğün **Private** olduğuna bak → oluştur. Kaggle yüklenen ZIP arşivlerini açar; defter hem klasörü hem ZIP'i kabul eder. ZIP ve içeriği toplamda yaklaşık 74 GB olduğundan Kaggle'daki özel veri kotanda yeterli boşluk olmalı.

İlk deneme her oturumun ilk **20 saniyesini** işler; FactorizePhys'i mevcut PURE başlangıç ağırlıklarından 3 epoch / 20 adım eğitir. Bu, veri ve eğitim hattının çalıştığını görmek için küçük bir pilot; bilimsel nihai kıyas değildir. Tam eğitim için süre, kişi bazlı test ayrımı ve sonuçlar görüldükten sonra ayarları büyütürüz.

Kaggle Notebook ayarları: Accelerator **None**; Internet **On** (kod deposunu almak için). Veri eklenince **Run All**.


In [ ]:
# 1. CPU ayarı — torch import edilmeden önce
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from pathlib import Path
import subprocess, sys, json, glob, shutil

assert not os.environ.get("CUDA_VISIBLE_DEVICES") == "", "CPU ayarı uygulanamadı"
WORK = Path("/kaggle/working")
REPO = Path("/tmp/rppg_physdrive_pilot_code")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Nisagwn/rppg.git", str(REPO)], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], check=True, capture_output=True, text=True).stdout.strip())


In [ ]:
# 2. Kaggle'daki koda PhysDrive hazırlama/eğitim desteğini ekle
# Bu kod, defter GitHub'a gönderilmeden de çalışsın diye hazırlayıcıyı defterin içine gömer.
from pathlib import Path
PREP_SCRIPT = '#!/usr/bin/env python\n"""PhysDrive RGB/BVP hazırlama: bir seferde bir video, CPU; yerel veya özel HF ZIP.\n\npython scripts/dl_prepare_physdrive.py --zip INPUT.zip --out data/dl --max-sec 60\nHF kaynağı: --hf-repo kullanici/depo --hf-file PhysDrive.zip (HF_TOKEN ortamda).\nEtiketler yazarların kare-indeksli BVP.mat düzeniyle okunur; zaman esnetilmez.\n"""\nimport argparse\nfrom contextlib import contextmanager\nimport hashlib\nimport io\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport random\nimport shutil\nimport tempfile\nfrom zipfile import ZipFile\nimport zlib\n\nimport cv2\nimport numpy as np\nfrom scipy.io import loadmat\n\nfrom dl_prepare import crop_video, label_polarity, save\ndef is_payload(name):\n    return not any(p == "__MACOSX" or p == ".DS_Store" or p.startswith("._")\n                   for p in PurePosixPath(name.replace("\\\\", "/")).parts)\n\n\nclass DirectoryInfo:\n    """Kaggle ZIP\'i açıp klasör olarak bağladığında aynı küçük arşiv arayüzü."""\n    def __init__(self, root, path):\n        self.filename = path.relative_to(root).as_posix()\n        stat = path.stat()\n        self.file_size = stat.st_size\n        # İçeriği 74 GB boyunca yeniden okumadan kararlı bir envanter izi.\n        self.CRC = (stat.st_size ^ stat.st_mtime_ns) & 0xFFFFFFFF\n\n\nclass DirectoryArchive:\n    def __init__(self, root):\n        self.root = Path(root)\n\n    def infolist(self):\n        return [DirectoryInfo(self.root, p) for p in self.root.rglob("*") if p.is_file()]\n\n    def getinfo(self, name):\n        path = self.root.joinpath(*PurePosixPath(name).parts)\n        if not path.is_file():\n            raise FileNotFoundError(name)\n        return DirectoryInfo(self.root, path)\n\n    def open(self, name):\n        return self.root.joinpath(*PurePosixPath(name).parts).open("rb")\n\n    def read(self, name):\n        return self.root.joinpath(*PurePosixPath(name).parts).read_bytes()\n\n\n@contextmanager\ndef open_source(local=None, repo=None, filename=None):\n    if bool(local) == bool(repo):\n        raise ValueError("Yerel ZIP veya HF deposundan birini seçin")\n    if local:\n        local = Path(local)\n        if local.is_dir():\n            yield DirectoryArchive(local)\n        else:\n            with ZipFile(local, allowZip64=True) as archive:\n                yield archive\n    else:\n        from huggingface_hub import HfFileSystem\n        fs = HfFileSystem(token=os.environ.get("HF_TOKEN"))\n        with fs.open(f"datasets/{repo}/{filename}", "rb", block_size=1024 * 1024,\n                     cache_type="blockcache") as stream, ZipFile(stream, allowZip64=True) as archive:\n            yield archive\n\n\ndef sessions(archive):\n    found = []\n    for info in archive.infolist():\n        if not is_payload(info.filename) or not info.filename.endswith("/Video/RGB.mp4"):\n            continue\n        stem = info.filename[:-len("/Video/RGB.mp4")]\n        person, scene = PurePosixPath(stem).parts[-2:]\n        found.append({"stem": stem, "person": person, "scene": scene, "video": info.filename})\n    keys = [(s["person"], s["scene"]) for s in found]\n    if len(keys) != len(set(keys)):\n        raise ValueError("Tekrarlanan katılımcı/oturum; veri sürümü incelenmeli")\n    return sorted(found, key=lambda s: (s["person"], s["scene"]))\n\n\ndef person_splits(people):\n    people = sorted(set(people))\n    if len(people) < 12:\n        raise ValueError("Tam PhysDrive katılımcı listesi bekleniyor; en az 12 kişi gerekli")\n    shuffled = list(people)\n    random.Random(42).shuffle(shuffled)  # Eğitim tohumundan bağımsız sabit kişi ayrımı.\n    test = set(shuffled[:8])\n    result = {p: ("test" if p in test else\n                  "validation" if zlib.crc32(("physdrive_" + p).encode()) % 10 == 0 else "train") for p in people}\n    if set(result.values()) != {"train", "validation", "test"}:\n        raise ValueError("Eğitim/doğrulama/test kişi grupları boş kalmamalı")\n    return result\n\n\ndef read_bvp(archive, stem):\n    path = stem + "/Label/BVP.mat"\n    if archive.getinfo(path).file_size > 10_000_000:\n        raise ValueError("Beklenmeyen etiket boyutu")\n    signal = np.asarray(loadmat(io.BytesIO(archive.read(path)))["BVP"], dtype=np.float32).ravel()\n    if len(signal) < 600 or not np.isfinite(signal).all() or np.std(signal) < 1e-6:\n        raise ValueError("Eksik, NaN veya düz BVP")\n    return signal\n\n\ndef aligned_prefix(n_frames, fps, bvp, seconds):\n    if not np.isfinite(fps) or abs(fps - 30) > 0.05:\n        raise ValueError("Kare-indeksli etiket için doğrulanmış 30 fps bekleniyor")\n    if abs(n_frames - len(bvp)) > 30:\n        raise ValueError("Video/BVP uzunluk farkı 30 kareyi aşıyor; zamanlama ayrıca incelenmeli")\n    n = min(n_frames, len(bvp), int(seconds * 30))\n    clip = bvp[:n]\n    if n < 600 or not np.isfinite(clip).all() or np.std(clip) < 1e-6:\n        raise ValueError("İlk kesit kısa veya BVP düz")\n    return n, clip\n\n\ndef prepare(archive, out, seconds=60, limit=0):\n    if seconds < 20 or limit < 0:\n        raise ValueError("En az 20 saniye ve negatif olmayan limit gerekli")\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    items = sessions(archive)\n    if not items:\n        raise ValueError("Video/RGB.mp4 düzeninde PhysDrive oturumu bulunamadı")\n    split = person_splits(s["person"] for s in items)\n    payload = [(s["video"], archive.getinfo(s["video"]).CRC,\n                archive.getinfo(s["stem"] + "/Label/BVP.mat").CRC) for s in items]\n    fingerprint = hashlib.sha256(json.dumps(payload).encode()).hexdigest()\n    config = {"source_fingerprint": fingerprint, "max_sec": seconds, "person_splits": split,\n              "label_timebase": "released per-frame BVP, 30 Hz; trim tail, no duration stretching"}\n    manifest_path = out / "physdrive_manifest.json"\n    if manifest_path.exists():\n        previous = json.loads(manifest_path.read_text(encoding="utf-8"))\n        if previous["config"] != config:\n            raise ValueError("Önceki veri hazırlama ayarları farklı; ayrı bir çıktı klasörü kullanın")\n    rows = []\n    with tempfile.TemporaryDirectory(prefix="physdrive_") as temp:\n        video_path = Path(temp, "RGB.mp4")\n        for index, item in enumerate(items[:limit] if limit else items, 1):\n            person, scene = item["person"], item["scene"]\n            partition = split[person]\n            folder = "test_physdrive" if partition == "test" else "physdrive"\n            dst = out / folder / f"physdrive_{person}_{scene}.npz"\n            record = {"person": person, "scene": scene, "split": partition}\n            if dst.exists() and dst.with_suffix(".npy").exists():\n                with np.load(dst) as meta:\n                    record.update(status="ready", frames=int(meta["n"]), face_rate=float(meta["face_rate"]))\n            else:\n                try:\n                    bvp = read_bvp(archive, item["stem"])\n                    required = archive.getinfo(item["video"]).file_size\n                    if shutil.disk_usage(temp).free < required + 2_000_000_000:\n                        raise OSError("Video geçici dosyası için yeterli boş alan yok")\n                    with archive.open(item["video"]) as src, video_path.open("wb") as target:\n                        shutil.copyfileobj(src, target, length=1024 * 1024)\n                    cap = cv2.VideoCapture(str(video_path))\n                    n_frames, fps = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), cap.get(cv2.CAP_PROP_FPS)\n                    cap.release()\n                    n, bvp = aligned_prefix(n_frames, fps, bvp, seconds)\n                    frames, face_rate = crop_video(str(video_path), max_frames=n, det_scale=2)\n                    if len(frames) != n:\n                        raise ValueError("Video çözücü beklenenden az kare döndürdü")\n                    if face_rate < 0.5:\n                        raise ValueError("Kesitte yüz bulunma oranı %50 altında")\n                    polarity, correlation = label_polarity(frames, bvp)\n                    if not np.isfinite(correlation):\n                        raise ValueError("Etiket işaret kontrolü sonlu değil")\n                    # Kaynak düzeyindeki işaret oylamasına yalnız eğitim kişileri katılır.\n                    save(str(dst), frames, bvp, face_rate, source="physdrive", camera="RGB", step=scene,\n                         person="physdrive_" + person, split=partition, polarity=polarity,\n                         polarity_corr=correlation if partition == "train" else 0.0,\n                         diagnostic_polarity_corr=correlation, original_frames=n_frames,\n                         original_label_samples=len(read_bvp(archive, item["stem"])))\n                    record.update(status="prepared", frames=n, face_rate=face_rate)\n                except ValueError as exc:\n                    record.update(status="excluded", reason=str(exc))\n                finally:\n                    if video_path.exists():\n                        video_path.unlink()\n            rows.append(record)\n            manifest = {"config": config, "complete": len(rows) == len(items),\n                        "total_sessions": len(items), "records": rows}\n            manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")\n            print(f"[{index}/{len(items)}] {person}/{scene}: {record[\'status\']}", flush=True)\n    return manifest\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    source = parser.add_mutually_exclusive_group(required=True)\n    source.add_argument("--zip")\n    source.add_argument("--hf-repo")\n    parser.add_argument("--hf-file", default="PhysDrive(Pub).zip")\n    parser.add_argument("--out", required=True)\n    parser.add_argument("--max-sec", type=int, default=60)\n    parser.add_argument("--limit", type=int, default=0, help="Yalnız CPU duman testi; tam eğitim girdisi sayılmaz")\n    args = parser.parse_args()\n    with open_source(args.zip, args.hf_repo, args.hf_file) as archive:\n        prepare(archive, args.out, args.max_sec, args.limit)\n\n\nif __name__ == "__main__":\n    main()\n'
(REPO / "scripts" / "dl_prepare_physdrive.py").write_text(PREP_SCRIPT, encoding="utf-8")
train_file = REPO / "scripts" / "dl_train.py"
train_text = train_file.read_text(encoding="utf-8")
if '"physdrive"' not in train_text:
    line = next((x for x in train_text.splitlines() if x.startswith("TRAIN_SOURCES = ")), None)
    if line is None or '"dlcn"' not in line:
        raise RuntimeError("Bu GitHub sürümünde TRAIN_SOURCES satırını bulamadım; defteri durdurdum.")
    train_text = train_text.replace(line, line.replace('"dlcn")', '"dlcn", "physdrive")'))
    train_file.write_text(train_text, encoding="utf-8")

# Kaggle imajında eksik olabilecek CPU paketleri
import importlib.util
missing = [name for name, mod in [("scipy", "scipy"), ("opencv-python-headless", "cv2")]
           if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
toolbox = REPO / "external" / "rPPG-Toolbox"
if not (toolbox / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "clone", "--depth", "1",
                    "https://github.com/ubicomplab/rPPG-Toolbox.git", str(REPO / "external" / "rPPG-Toolbox")], check=True)
print("PhysDrive ön hazırlayıcı ve CPU eğitim kodu hazır.")


In [ ]:
# 3. Girdi klasörünü bul (Kaggle ZIP'i açtıysa klasörü, açmadıysa ZIP'i kullanır)
INPUT_ROOT = Path("/kaggle/input")
zip_candidates = [p for p in INPUT_ROOT.rglob("*.zip") if "physdrive" in p.name.lower()]
publication_dirs = [p for p in INPUT_ROOT.rglob("PhysDrive(Publication)") if p.is_dir()]
if len(publication_dirs) == 1:
    SOURCE_PATH = publication_dirs[0].parent
elif len(zip_candidates) == 1:
    SOURCE_PATH = zip_candidates[0]
else:
    raise RuntimeError(
        f"PhysDrive girdisi tek olmalı. bulunan ZIP={len(zip_candidates)}, "
        f"Publication klasörü={len(publication_dirs)}. Kaggle Notebook > Add Input ile Private PhysDrive datasetini bağla."
    )
print("Girdi:", SOURCE_PATH)
if SOURCE_PATH.is_file():
    print(f"ZIP boyutu: {SOURCE_PATH.stat().st_size / 1e9:.2f} GB")
else:
    print("Kaggle ZIP'i klasöre açmış; veriler bu klasörden okunacak.")


In [ ]:
# 4. Oturumları CPU'da işle; ilk 20 saniye / oturum
os.chdir(REPO)
DATA = Path("/tmp/physdrive_pilot_data")
OUT = WORK / "cikti" / "physdrive_cpu_pilot"
MAX_SEC = 20
cmd = [sys.executable, "-u", "scripts/dl_prepare_physdrive.py",
       "--zip", str(SOURCE_PATH), "--out", str(DATA), "--max-sec", str(MAX_SEC)]
print("Komut:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": "-1", "PYTHONUNBUFFERED": "1"})
manifest = json.loads((DATA / "physdrive_manifest.json").read_text(encoding="utf-8"))
rows = manifest["records"]
from collections import Counter
print("Durum:", dict(Counter(x["status"] for x in rows)))
print("Bölüm:", dict(Counter(x["split"] for x in rows if x["status"] in ("ready", "prepared"))))
print("Manifest:", DATA / "physdrive_manifest.json")


In [ ]:
# 5. Küçük CPU ince ayarı. Bu hücre gerçekten eğitimi başlatır.
# GPU ortam değişkeniyle kapalı ve CPU cihazı ayrıca doğrulanıyor.
import torch
assert not torch.cuda.is_available(), "GPU görünür durumda; CPU dışına çıkmamak için eğitimi durdurdum."
prepared = [x for x in rows if x["status"] in ("ready", "prepared")]
train_n = sum(x["split"] == "train" for x in prepared)
val_n = sum(x["split"] == "validation" for x in prepared)
if train_n < 4 or val_n < 2:
    raise RuntimeError(f"Pilot için az veri: train={train_n}, validation={val_n}. Hazırlama manifestini incele.")
OUT.mkdir(parents=True, exist_ok=True)
train_cmd = [sys.executable, "-u", "scripts/dl_train.py", "--data", str(DATA), "--out", str(OUT),
             "--arch", "factorizephys", "--source-weights", "physdrive=1",
             "--epochs", "3", "--steps", "20", "--bs", "2", "--lr", "5e-5",
             "--val-max", "16", "--seed", "42"]
print("CPU eğitimi başlıyor:", " ".join(train_cmd), flush=True)
subprocess.run(train_cmd, check=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": "-1", "PYTHONUNBUFFERED": "1"})
print("Eğitim çıktı klasörü:", OUT)
print("Çıktılar:", sorted(p.name for p in OUT.iterdir()))


## Çıktılar

Başarılı olursa `/kaggle/working/cikti/physdrive_cpu_pilot/` içinde `best.pth`, `last.pth` ve eğitim geçmişi oluşur. Kaggle'da sağdaki **Output** bölümünden dosyaları indir veya notebook'un **Save Version** ekranında output'u sakla. Bu hücre veriyi veya ağırlıkları Hugging Face'e/GitHub'a yüklemez.

Pilotun sonunda eğitim ve doğrulama video/kişi sayıları, dışlanan oturumlar ve başlangıç/bitiş doğrulama MAE'si görülmeli. Eğitim başarısı ve veri senkronu onaylanmadan tüm oturum/süre için uzun koşuya geçme.
